In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load all datasets
laps = pd.read_csv('../data/processed/f1_race_laps.csv')
qual = pd.read_csv('../data/processed/f1_qualifying.csv')
cars = pd.read_csv('../data/processed/cars_unified.csv')

print("Laps:", laps.shape)
print("Qual:", qual.shape)
print("Cars:", cars.shape)
print("\nLaps columns:", list(laps.columns))

Laps: (73398, 24)
Qual: (1039, 14)
Cars: (18392, 21)

Laps columns: ['Driver', 'DriverNumber', 'Team', 'LapTime', 'LapNumber', 'Compound', 'TyreLife', 'FreshTyre', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'PitInTime', 'PitOutTime', 'TrackStatus', 'IsAccurate', 'GP', 'Location', 'Year', 'Round', 'HasPitStop']


In [2]:
# ── Cell 2: F1 feature engineering ──

# Work on clean dry laps only
f1 = laps[
    laps['Compound'].isin(['SOFT', 'MEDIUM', 'HARD']) &
    laps['IsAccurate'].eq(True) &
    laps['LapTime'].notna()
].copy()

# 1. Tyre age buckets
f1['TyreAge_bucket'] = pd.cut(
    f1['TyreLife'],
    bins=[0, 5, 15, 30, 100],
    labels=['new', 'prime', 'used', 'worn']
)

# 2. Compound encoded
compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}
f1['Compound_enc'] = f1['Compound'].map(compound_map)

# 3. Speed trap efficiency — how fast relative to circuit average
circuit_avg_speed = f1.groupby('GP')['SpeedST'].transform('mean')
f1['SpeedST_norm'] = f1['SpeedST'] / circuit_avg_speed

# 4. Sector balance — ratio of S1 to total lap (aero sensitivity proxy)
# High S1 ratio = more time in technical sections = higher downforce circuit
f1['S1_ratio'] = f1['Sector1Time'] / f1['LapTime']
f1['S2_ratio'] = f1['Sector2Time'] / f1['LapTime']
f1['S3_ratio'] = f1['Sector3Time'] / f1['LapTime']

# 5. Lap time delta from circuit median (performance vs field)
circuit_median = f1.groupby('GP')['LapTime'].transform('median')
f1['LapTime_delta'] = f1['LapTime'] - circuit_median

# 6. Pit stop flag already exists as HasPitStop

# 7. Year as relative (2022=0, 2023=1, 2024=2)
f1['Year_rel'] = f1['Year'] - 2022

# 8. Team encoded
f1['Team_enc'] = f1['Team'].astype('category').cat.codes

print("F1 features engineered:")
print(f"  Rows: {len(f1):,}")
new_cols = ['TyreAge_bucket','Compound_enc','SpeedST_norm',
            'S1_ratio','S2_ratio','S3_ratio',
            'LapTime_delta','Year_rel','Team_enc']
print(f"  New features: {new_cols}")
print(f"\nNull check:")
print(f1[new_cols].isnull().sum())

F1 features engineered:
  Rows: 60,366
  New features: ['TyreAge_bucket', 'Compound_enc', 'SpeedST_norm', 'S1_ratio', 'S2_ratio', 'S3_ratio', 'LapTime_delta', 'Year_rel', 'Team_enc']

Null check:
TyreAge_bucket    149
Compound_enc        0
SpeedST_norm        0
S1_ratio            0
S2_ratio            0
S3_ratio            0
LapTime_delta       0
Year_rel            0
Team_enc            0
dtype: int64


In [3]:
# ── Cell 3: Cars feature engineering ──

c = cars.copy()

# 1. Aero efficiency score — MPG per unit drag
# Higher = more aerodynamically efficient for its fuel consumption
# Only for rows with Cd data
c['aero_efficiency'] = np.where(
    c['cd'].notna(),
    c['highway_mpg'] / c['cd'],
    np.nan
)

# 2. Engine power proxy — displacement per cylinder
c['displacement_per_cyl'] = c['engine_displacement_l'] / c['cylinders']

# 3. City/highway split ratio — higher = more highway biased
c['hwy_city_ratio'] = c['highway_mpg'] / c['city_mpg']

# 4. Fuel type simplified
fuel_simple = {
    'Regular': 'regular',
    'Premium': 'premium',
    'Midgrade': 'regular',
    'Diesel': 'diesel',
    'Regular Gas or Electricity': 'hybrid',
    'Premium Gas or Electricity': 'hybrid',
    'Regular Gas and Electricity': 'hybrid',
    'Premium Gas and Electricity': 'hybrid',
}
c['fuel_simple'] = c['fuel_type'].map(fuel_simple)
c['fuel_enc'] = c['fuel_simple'].astype('category').cat.codes

# 5. Vehicle class simplified
c['class_simple'] = c['vehicle_class'].str.extract(
    r'(Compact|Midsize|Large|SUV|Pickup|Minivan|Wagon|Subcompact|Two Seater|Van)',
    expand=False
).fillna('Other')

# 6. CO2 efficiency (lower = cleaner)
c['co2_per_mpg'] = c['co2_grams_per_mile'] / c['combined_mpg']

# 7. Year relative
c['year_rel'] = c['year'] - 2010

new_cols = ['aero_efficiency','displacement_per_cyl','hwy_city_ratio',
            'fuel_enc','co2_per_mpg','year_rel']

print("Cars features engineered:")
print(f"  Rows: {len(c):,}")
print(f"  New features: {new_cols}")
print(f"\nNull check:")
print(c[new_cols].isnull().sum())
print(f"\nAero efficiency stats (Cd rows only):")
print(c['aero_efficiency'].describe().round(2))

Cars features engineered:
  Rows: 18,392
  New features: ['aero_efficiency', 'displacement_per_cyl', 'hwy_city_ratio', 'fuel_enc', 'co2_per_mpg', 'year_rel']

Null check:
aero_efficiency         17840
displacement_per_cyl        0
hwy_city_ratio              0
fuel_enc                    0
co2_per_mpg              2939
year_rel                    0
dtype: int64

Aero efficiency stats (Cd rows only):
count    552.00
mean     105.73
std       49.73
min       41.85
25%       68.75
50%       83.62
75%      145.00
max      285.71
Name: aero_efficiency, dtype: float64


In [4]:
# ── Cell 4: Win probability features ──

q = qual.copy()

# 1. Best qualifying time across Q1/Q2/Q3
q['BestQual'] = q[['Q1','Q2','Q3']].min(axis=1)

# 2. Gap to pole per race
q['GapToPole'] = q.groupby(['GP','Year'])['BestQual'].transform(
    lambda x: x - x.min()
)

# 3. Qualifying position (rank within session)
q['QualPos'] = q.groupby(['GP','Year'])['BestQual'].rank(method='min')

# 4. Team average qualifying pace (last 3 races rolling)
q = q.sort_values(['Year','Round'])
q['TeamAvgQual'] = (q.groupby(['TeamName','Year'])['BestQual']
                    .transform(lambda x: x.rolling(3, min_periods=1).mean()))

# 5. Recent form — driver avg gap to pole last 3 races
q['RecentForm'] = (q.groupby(['Abbreviation','Year'])['GapToPole']
                   .transform(lambda x: x.rolling(3, min_periods=1).mean()))

# 6. Front row flag (pole or P2)
q['FrontRow'] = (q['QualPos'] <= 2).astype(int)

# 7. Top 5 flag
q['Top5'] = (q['QualPos'] <= 5).astype(int)

new_cols = ['BestQual','GapToPole','QualPos',
            'TeamAvgQual','RecentForm','FrontRow','Top5']

print("Qualifying features engineered:")
print(f"  Rows: {len(q):,}")
print(f"\nNull check:")
print(q[new_cols].isnull().sum())
print(f"\nSample:")
print(q[['Abbreviation','TeamName','GP','Year','QualPos','GapToPole','RecentForm']].head(10).to_string())

Qualifying features engineered:
  Rows: 1,039

Null check:
BestQual       10
GapToPole      10
QualPos        10
TeamAvgQual     0
RecentForm      0
FrontRow        0
Top5            0
dtype: int64

Sample:
  Abbreviation         TeamName                  GP  Year  QualPos  GapToPole  RecentForm
0          LEC          Ferrari  Bahrain Grand Prix  2022      1.0      0.000       0.000
1          VER  Red Bull Racing  Bahrain Grand Prix  2022      2.0      0.123       0.123
2          SAI          Ferrari  Bahrain Grand Prix  2022      3.0      0.129       0.129
3          PER  Red Bull Racing  Bahrain Grand Prix  2022      4.0      0.363       0.363
4          HAM         Mercedes  Bahrain Grand Prix  2022      5.0      0.490       0.490
5          BOT       Alfa Romeo  Bahrain Grand Prix  2022      8.0      1.002       1.002
6          MAG     Haas F1 Team  Bahrain Grand Prix  2022      7.0      0.903       0.903
7          ALO           Alpine  Bahrain Grand Prix  2022      9.0      1

In [5]:
# ── Cell 5: Save engineered datasets ──

# F1 laps — drop rows with null TyreAge
f1_clean = f1.dropna(subset=['TyreAge_bucket'])

# Select final ML-ready columns
f1_ml = f1_clean[[
    'Driver', 'Team', 'GP', 'Year', 'Round',
    'LapTime', 'LapNumber', 'Compound', 'Compound_enc',
    'TyreLife', 'TyreAge_bucket',
    'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'SpeedST_norm',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'S1_ratio', 'S2_ratio', 'S3_ratio',
    'LapTime_delta', 'HasPitStop', 'Year_rel', 'Team_enc'
]].copy()

# Cars — select final columns
cars_ml = c[[
    'make', 'model', 'year', 'vehicle_class', 'class_simple',
    'engine_displacement_l', 'cylinders', 'displacement_per_cyl',
    'fuel_type', 'fuel_simple', 'fuel_enc',
    'city_mpg', 'highway_mpg', 'combined_mpg',
    'hwy_city_ratio', 'co2_grams_per_mile', 'co2_per_mpg',
    'cd', 'aero_efficiency', 'year_rel'
]].copy()

# Qualifying — select final columns
qual_ml = q[[
    'Abbreviation', 'FullName', 'TeamName',
    'GP', 'Year', 'Round',
    'BestQual', 'GapToPole', 'QualPos',
    'TeamAvgQual', 'RecentForm', 'FrontRow', 'Top5',
    'Q1', 'Q2', 'Q3'
]].dropna(subset=['BestQual']).copy()

# Save
f1_ml.to_csv('../data/processed/f1_ml_ready.csv', index=False)
cars_ml.to_csv('../data/processed/cars_ml_ready.csv', index=False)
qual_ml.to_csv('../data/processed/qual_ml_ready.csv', index=False)

print("Saved:")
print(f"  f1_ml_ready.csv   — {len(f1_ml):,} rows, {f1_ml.shape[1]} cols")
print(f"  cars_ml_ready.csv — {len(cars_ml):,} rows, {cars_ml.shape[1]} cols")
print(f"  qual_ml_ready.csv — {len(qual_ml):,} rows, {qual_ml.shape[1]} cols")
print("\nFeature engineering complete — ready for modeling")

Saved:
  f1_ml_ready.csv   — 60,217 rows, 26 cols
  cars_ml_ready.csv — 18,392 rows, 20 cols
  qual_ml_ready.csv — 1,029 rows, 16 cols

Feature engineering complete — ready for modeling
